In [1]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from pathlib import Path

In [3]:
DATASET = "../dataset/splitted"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_CLASSES = 6
EPOCHS = 10
BATCH_SIZE = 16
IMGSZ = 384
LR = 1e-4

In [4]:
transform_train = transforms.Compose([
    transforms.Resize((IMGSZ, IMGSZ)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

transform_test = transforms.Compose([
    transforms.Resize((IMGSZ, IMGSZ)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

In [5]:
train_dataset = datasets.ImageFolder(f"{DATASET}/train", transform=transform_train)
test_dataset  = datasets.ImageFolder(f"{DATASET}/test",  transform=transform_test)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader   = DataLoader(test_dataset,  batch_size=BATCH_SIZE)

print(f"Classes: {train_dataset.classes}")

Classes: ['Gilmore', 'Golden Boy', 'Golden Boy x Gilmore', 'Golden Boy x Harold Brown', 'Harold Brown', 'Kearny']


In [6]:
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
model = model.to(DEVICE)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to C:\Users\geral/.cache\torch\hub\checkpoints\resnet50-11ad3fa6.pth


100.0%


In [7]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3)
criterion = nn.CrossEntropyLoss()

In [9]:
best_acc = 0.0

for epoch in range(EPOCHS):
    # Training
    model.train()
    train_loss, correct, total = 0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    train_acc = correct / total

    # Evaluation
    model.eval()
    val_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            outputs = model(imgs)
            val_loss += criterion(outputs, labels).item()
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
    val_acc = correct / total
    scheduler.step(val_loss)

    print(f"Epoch [{epoch+1}/{EPOCHS}] "
          f"Train Loss: {train_loss/len(train_loader):.4f} | Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss/len(test_loader):.4f} | Val Acc: {val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "resnet50_best.pt")
        print(f"  ✅ Saved best model with val acc: {best_acc:.4f}")

print(f"\nBest Val Accuracy: {best_acc:.4f}")

Epoch [1/10] Train Loss: 0.3024 | Train Acc: 0.9362 | Val Loss: 0.0065 | Val Acc: 1.0000
  ✅ Saved best model with val acc: 1.0000
Epoch [2/10] Train Loss: 0.0100 | Train Acc: 0.9990 | Val Loss: 0.0005 | Val Acc: 1.0000
Epoch [3/10] Train Loss: 0.0066 | Train Acc: 0.9986 | Val Loss: 0.0003 | Val Acc: 1.0000
Epoch [4/10] Train Loss: 0.0061 | Train Acc: 0.9990 | Val Loss: 0.0009 | Val Acc: 1.0000
Epoch [5/10] Train Loss: 0.0030 | Train Acc: 0.9995 | Val Loss: 0.0002 | Val Acc: 1.0000
Epoch [6/10] Train Loss: 0.0040 | Train Acc: 0.9995 | Val Loss: 0.0017 | Val Acc: 0.9989
Epoch [7/10] Train Loss: 0.0014 | Train Acc: 1.0000 | Val Loss: 0.0001 | Val Acc: 1.0000
Epoch [8/10] Train Loss: 0.0007 | Train Acc: 1.0000 | Val Loss: 0.0001 | Val Acc: 1.0000
Epoch [9/10] Train Loss: 0.0008 | Train Acc: 1.0000 | Val Loss: 0.0001 | Val Acc: 1.0000
Epoch [10/10] Train Loss: 0.0005 | Train Acc: 1.0000 | Val Loss: 0.0000 | Val Acc: 1.0000

Best Val Accuracy: 1.0000
